# Understanding Context Engineering: Stateless vs. Stateful Agents

This notebook accompanies the article "Understanding Context Engineering: Why Your AI Agent Keeps Forgetting (And How to Fix It)"

You'll learn:
1. Why LLMs are inherently stateless
2. **Context Engineering as a systematic discipline** *(New)*
3. **The four layers of context** *(New)*
4. How Google ADK Sessions solve the stateless problem
5. **Memory taxonomy: semantic, episodic, procedural** *(New)*
6. How Sessions, State, and Memory work together

---

## Setup

First, let's install and import the required packages.

In [1]:
# Install required packages if not already installed
!pip install google-adk python-dotenv -q


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [1]:
import os
from pathlib import Path
from google import genai
from google.genai import types
from dotenv import load_dotenv

# Load environment variables from .env file
# This looks for .env in the project root (parent of notebooks/)
env_path = Path(__file__).parent.parent / ".env" if "__file__" in globals() else Path("../.env")
load_dotenv(dotenv_path=env_path)

# Get API key from environment
api_key = os.getenv("GOOGLE_API_KEY")

if not api_key:
    raise ValueError(
        "GOOGLE_API_KEY not found! Please:\n"
        "1. Copy .env.example to .env\n"
        "2. Add your API key to .env\n"
        "3. Get your key from: https://aistudio.google.com/apikey"
    )

os.environ["GOOGLE_API_KEY"] = api_key

# Initialize the client
client = genai.Client()
MODEL_ID = "gemini-2.5-flash"

print("✅ Environment loaded successfully")
print(f"🔑 API key loaded: {api_key[:2]}...{api_key[-2:]}")  # Show only first 8 and last 4 chars

✅ Environment loaded successfully
🔑 API key loaded: AI...go


## Part 1: The Stateless Problem

Let's demonstrate why LLMs "forget" between calls. Each API call is completely independent—the model has no memory of previous interactions.

In [2]:
def stateless_call(message: str) -> str:
    """Make a stateless API call - no conversation history."""
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=message
    )
    return response.text

In [5]:
# First message - introduce ourselves
response1 = stateless_call("My name is Alex and I'm a software engineer who loves hiking.")
print("User: My name is Alex and I'm a software engineer who loves hiking.")
print(f"Agent: {response1}")

User: My name is Alex and I'm a software engineer who loves hiking.
Agent: Hi Alex! That's a great combination. Sounds like you get a good balance between the digital world and the great outdoors.

It's common for software engineers to love hiking – a perfect way to clear the head and escape the screens!

What kind of hiking do you usually enjoy? Or what's your favorite part about hitting the trails?


In [6]:
# Second message - ask about what we just said
response2 = stateless_call("What's my name and what do I do for work?")
print("User: What's my name and what do I do for work?")
print(f"Agent: {response2}")

User: What's my name and what do I do for work?
Agent: As an AI, I don't know your name or what you do for work. I don't have access to personal information about you.

If you'd like to tell me, I'm happy to know! Otherwise, feel free to ask me anything else.


### 🔴 The Problem

The agent has no idea who you are! Each API call is independent—the model doesn't retain any information from the previous call.

This is the **stateless problem**: LLMs don't have built-in memory between requests.